# MovieLens-100K Baselines (Single Notebook)

This notebook runs baseline models end-to-end in one place: data preparation, model training, evaluation, and artifacts export.

It is designed to be similar to this repository's baseline setup while remaining fully self-contained in a single notebook.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/faroukq1/LLM-Sequential-Recommendation.git"
PREFERRED_REPO_DIR = Path("/kaggle/working/LLM-Sequential-Recommendation")
cwd = Path.cwd()

if (cwd / "main").exists():
    repo_root = cwd
elif (cwd.parent / "main").exists():
    repo_root = cwd.parent
else:
    if not PREFERRED_REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PREFERRED_REPO_DIR)], check=True)
    repo_root = PREFERRED_REPO_DIR

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repo root:", repo_root)

In [ ]:
import logging
import math
import pickle
import random
import tarfile
import time
import urllib.request
import zipfile
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s: %(message)s")
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Paths
WORK_DIR = Path("/kaggle/working/llmseqrec_ml100k_single_notebook")
DATA_DIR = WORK_DIR / "data"
RECS_DIR = WORK_DIR / "recommendations"
RESULTS_DIR = WORK_DIR / "results"

for p in [WORK_DIR, DATA_DIR, RECS_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Data settings
DATASET_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
SESSION_GAP_MINUTES = 30
MIN_SESSION_LEN = 2
TEST_FRAC = 0.2
NUM_FOLDS = 0
FILTER_NON_TRAINED_TEST_ITEMS = True
TOP_KS = [10, 20]

# Model selection
INCLUDE_MODELS = [
    "Popular",
    "SKNN",
    "S-SKNN",
    "SF-SKNN",
    "V-SKNN",
    "GRU4Rec",
    "BERT4Rec",
    "SASRec",
]

# Shared compute settings
CORES = 2
EVAL_CORES = 1
VERBOSE = False

# SKNN settings
SKNN_K = 200
SKNN_SAMPLE_SIZE = 1000

# Neural settings
NUM_EPOCHS = 6
FIT_BATCH_SIZE = 256
PRED_BATCH_SIZE = 4096
TRAIN_VAL_FRACTION = 0.1
EARLY_STOPPING_PATIENCE = 1
MAX_SEQ_LEN = 20
NEURAL_EMB_DIM = 64
HIDDEN_DIM = 128
DROP_RATE = 0.2
MASK_PROB = 0.2
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001

print("Work dir:", WORK_DIR)
print("Models:", INCLUDE_MODELS)

In [ ]:
class BaseModel:
    def train(self, train_data: pd.DataFrame) -> None:
        raise NotImplementedError

    def predict(self, prompts: Dict[int, np.ndarray], top_k: int = 10) -> Dict[int, np.ndarray]:
        raise NotImplementedError

    def name(self) -> str:
        raise NotImplementedError


class SessionBasedPopular(BaseModel):
    def __init__(self) -> None:
        self.ranked_items: np.ndarray = np.array([], dtype=np.int64)

    def train(self, train_data: pd.DataFrame) -> None:
        self.ranked_items = train_data["ItemId"].value_counts().index.to_numpy(dtype=np.int64)

    def predict(self, prompts: Dict[int, np.ndarray], top_k: int = 10) -> Dict[int, np.ndarray]:
        if len(self.ranked_items) == 0:
            raise ValueError("Model not trained.")
        if top_k > len(self.ranked_items):
            top_k = len(self.ranked_items)

        return {sid: self.ranked_items[:top_k].copy() for sid in prompts}

    def name(self) -> str:
        return "Popularity"


class SessionBasedCF(BaseModel):
    def __init__(
        self,
        k: int = 200,
        sample_size: int = 1000,
        sampling: str = "recent",
        similarity_measure: str = "cosine",
        idf_weighting: bool = True,
        sequential_weighting: bool = False,
        sequential_filter: bool = False,
        decay: str | None = None,
        filter_prompt_items: bool = True,
    ) -> None:
        self.k = k
        self.sample_size = sample_size
        self.sampling = sampling
        self.similarity_measure = similarity_measure
        self.idf_weighting = idf_weighting
        self.sequential_weighting = sequential_weighting
        self.sequential_filter = sequential_filter
        self.decay = decay
        self.filter_prompt_items = filter_prompt_items

        self.session_items: Dict[int, np.ndarray] = {}
        self.session_last_time: Dict[int, int] = {}
        self.item_to_sessions: Dict[int, set[int]] = defaultdict(set)
        self.idf: Dict[int, float] = {}
        self.popular_items: List[int] = []

    def train(self, train_data: pd.DataFrame) -> None:
        df = train_data.sort_values(["SessionId", "Time", "ItemId"]).copy()
        self.popular_items = df["ItemId"].value_counts().index.astype(int).tolist()

        for sid, group in df.groupby("SessionId"):
            items = group["ItemId"].to_numpy(dtype=np.int64)
            self.session_items[int(sid)] = items

            ts = pd.to_datetime(group["Time"]).iloc[-1]
            self.session_last_time[int(sid)] = int(ts.value)

            for item in np.unique(items):
                self.item_to_sessions[int(item)].add(int(sid))

        if self.idf_weighting:
            n_sessions = max(1, len(self.session_items))
            self.idf = {
                item: math.log((1 + n_sessions) / (1 + len(sids))) + 1.0
                for item, sids in self.item_to_sessions.items()
            }

    def _prompt_item_weights(self, prompt: np.ndarray) -> Dict[int, float]:
        weights: Dict[int, float] = {}
        n = len(prompt)
        if n == 0:
            return weights

        for idx, item in enumerate(prompt):
            item = int(item)
            pos_from_end = n - idx
            w = 1.0

            if self.decay == "harmonic":
                w *= 1.0 / max(1, pos_from_end)
            elif self.decay == "linear":
                w *= (idx + 1) / n

            if self.sequential_weighting:
                w *= (idx + 1) / n

            if self.idf_weighting:
                w *= self.idf.get(item, 1.0)

            if item not in weights or w > weights[item]:
                weights[item] = w

        return weights

    def _sample_candidates(self, prompt: np.ndarray) -> List[int]:
        candidates: set[int] = set()
        for item in np.unique(prompt):
            candidates |= self.item_to_sessions.get(int(item), set())

        cand = list(candidates)
        if len(cand) == 0:
            return cand

        if self.sampling == "recent":
            cand.sort(key=lambda sid: self.session_last_time.get(sid, 0), reverse=True)
        else:
            random.shuffle(cand)

        return cand[: self.sample_size]

    def _similarity(self, prompt_weights: Dict[int, float], candidate_items: np.ndarray) -> float:
        if len(prompt_weights) == 0:
            return 0.0

        cand_set = set(int(i) for i in candidate_items)
        common = [item for item in prompt_weights if item in cand_set]
        if len(common) == 0:
            return 0.0

        numerator = sum(prompt_weights[item] for item in common)

        if self.similarity_measure == "dot":
            return float(numerator)

        denom_prompt = math.sqrt(sum(v * v for v in prompt_weights.values()))
        denom_cand = math.sqrt(max(1, len(cand_set)))
        denom = max(1e-12, denom_prompt * denom_cand)
        return float(numerator / denom)

    def _candidate_items_from_neighbor(self, neighbor_items: np.ndarray, prompt: np.ndarray) -> np.ndarray:
        if not self.sequential_filter:
            return neighbor_items

        if len(prompt) == 0:
            return np.array([], dtype=np.int64)

        last_item = int(prompt[-1])
        positions = np.where(neighbor_items == last_item)[0]
        if len(positions) == 0:
            return np.array([], dtype=np.int64)

        tails = [neighbor_items[p + 1 :] for p in positions if p + 1 < len(neighbor_items)]
        if len(tails) == 0:
            return np.array([], dtype=np.int64)
        return np.concatenate(tails).astype(np.int64)

    def predict(self, prompts: Dict[int, np.ndarray], top_k: int = 10) -> Dict[int, np.ndarray]:
        predictions: Dict[int, np.ndarray] = {}

        for sid, prompt in prompts.items():
            prompt = np.asarray(prompt, dtype=np.int64)
            prompt_weights = self._prompt_item_weights(prompt)
            candidate_sessions = self._sample_candidates(prompt)

            sims: List[Tuple[int, float]] = []
            for cand_sid in candidate_sessions:
                sim = self._similarity(prompt_weights, self.session_items[cand_sid])
                if sim > 0:
                    sims.append((cand_sid, sim))

            sims.sort(key=lambda x: x[1], reverse=True)
            neighbors = sims[: self.k]

            seen_items = set(int(i) for i in prompt)
            item_scores: Dict[int, float] = defaultdict(float)

            for neigh_sid, sim in neighbors:
                neigh_items = self.session_items[neigh_sid]
                cand_items = self._candidate_items_from_neighbor(neigh_items, prompt)
                for item in cand_items:
                    item = int(item)
                    if self.filter_prompt_items and item in seen_items:
                        continue
                    w = sim
                    if self.idf_weighting:
                        w *= self.idf.get(item, 1.0)
                    item_scores[item] += w

            ranked = sorted(item_scores, key=item_scores.get, reverse=True)

            # Fill with popular items to guarantee top_k length.
            if len(ranked) < top_k:
                for item in self.popular_items:
                    if self.filter_prompt_items and item in seen_items:
                        continue
                    if item not in item_scores and item not in ranked:
                        ranked.append(item)
                    if len(ranked) >= top_k:
                        break

            predictions[sid] = np.array(ranked[:top_k], dtype=np.int64)

        return predictions

    def name(self) -> str:
        if self.decay is not None:
            return "V-SKNN"
        if self.sequential_filter:
            return "SF-SKNN"
        if self.sequential_weighting:
            return "S-SKNN"
        return "SKNN"


class NeuralSequenceRecommender(BaseModel):
    def __init__(
        self,
        model_name: str,
        architecture: str,
        N: int = 20,
        emb_dim: int = 64,
        hidden_dim: int = 128,
        drop_rate: float = 0.2,
        num_epochs: int = 6,
        fit_batch_size: int = 256,
        pred_batch_size: int = 4096,
        learning_rate: float = 0.001,
        weight_decay: float = 1e-4,
        pred_seen: bool = False,
        verbose: bool = False,
    ) -> None:
        self.model_name = model_name
        self.architecture = architecture
        self.N = N
        self.emb_dim = emb_dim
        self.hidden_dim = hidden_dim
        self.drop_rate = drop_rate
        self.num_epochs = num_epochs
        self.fit_batch_size = fit_batch_size
        self.pred_batch_size = pred_batch_size
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.pred_seen = pred_seen
        self.verbose = verbose

        self.model: keras.Model | None = None
        self.item_to_idx: Dict[int, int] = {}
        self.idx_to_item: Dict[int, int] = {}
        self.popular_items: List[int] = []

    def _pad_left(self, seq: List[int]) -> np.ndarray:
        arr = np.zeros(self.N, dtype=np.int32)
        trimmed = seq[-self.N :]
        arr[-len(trimmed) :] = np.array(trimmed, dtype=np.int32)
        return arr

    def _build_train_examples(self, train_data: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        sessions = (
            train_data.sort_values(["SessionId", "Time", "ItemId"])
            .groupby("SessionId")["ItemId"]
            .apply(list)
            .tolist()
        )

        x_list: List[np.ndarray] = []
        y_list: List[int] = []

        for session in sessions:
            mapped = [self.item_to_idx[int(item)] for item in session if int(item) in self.item_to_idx]
            if len(mapped) < 2:
                continue

            for t in range(1, len(mapped)):
                prefix = mapped[:t]
                x_list.append(self._pad_left(prefix))
                y_list.append(mapped[t])

        if len(x_list) == 0:
            raise ValueError("No train examples were generated.")

        return np.vstack(x_list).astype(np.int32), np.array(y_list, dtype=np.int32)

    def _build_model(self, num_items: int) -> keras.Model:
        inp = keras.Input(shape=(self.N,), dtype="int32")
        item_emb = keras.layers.Embedding(num_items + 1, self.emb_dim)(inp)

        if self.architecture == "gru":
            x = keras.layers.GRU(self.hidden_dim)(item_emb)
            x = keras.layers.Dropout(self.drop_rate)(x)
        else:
            # Minimal transformer-style encoder for BERT/SAS baselines.
            pos_idx = tf.range(start=0, limit=self.N, delta=1)
            pos_emb = keras.layers.Embedding(self.N, self.emb_dim)(pos_idx)
            x = item_emb + pos_emb

            attn = keras.layers.MultiHeadAttention(
                num_heads=2,
                key_dim=max(1, self.emb_dim // 2),
                dropout=self.drop_rate,
            )(x, x, use_causal_mask=(self.architecture == "sas"))

            x = keras.layers.LayerNormalization(epsilon=1e-6)(x + attn)
            ff = keras.layers.Dense(self.emb_dim * 2, activation="relu")(x)
            ff = keras.layers.Dense(self.emb_dim)(ff)
            x = keras.layers.LayerNormalization(epsilon=1e-6)(x + ff)
            x = keras.layers.Lambda(lambda t: t[:, -1, :])(x)
            x = keras.layers.Dropout(self.drop_rate)(x)

        out = keras.layers.Dense(num_items + 1, activation="softmax")(x)
        model = keras.Model(inp, out)

        if hasattr(keras.optimizers, "AdamW"):
            optimizer = keras.optimizers.AdamW(
                learning_rate=self.learning_rate,
                weight_decay=self.weight_decay,
            )
        else:
            optimizer = keras.optimizers.Adam(learning_rate=self.learning_rate)

        model.compile(
            optimizer=optimizer,
            loss=keras.losses.SparseCategoricalCrossentropy(),
        )
        return model

    def train(self, train_data: pd.DataFrame) -> None:
        item_counts = train_data["ItemId"].value_counts()
        items = item_counts.index.astype(int).tolist()
        self.popular_items = items

        self.item_to_idx = {item: idx + 1 for idx, item in enumerate(items)}
        self.idx_to_item = {idx + 1: item for idx, item in enumerate(items)}

        x_train, y_train = self._build_train_examples(train_data)
        self.model = self._build_model(num_items=len(items))

        self.model.fit(
            x_train,
            y_train,
            epochs=self.num_epochs,
            batch_size=self.fit_batch_size,
            validation_split=0.1 if len(x_train) >= 100 else 0.0,
            verbose=2 if self.verbose else 0,
        )

    def predict(self, prompts: Dict[int, np.ndarray], top_k: int = 10) -> Dict[int, np.ndarray]:
        if self.model is None:
            raise ValueError("Model not trained.")

        keys = list(prompts.keys())
        x_list: List[np.ndarray] = []
        seen_idx_list: List[set[int]] = []
        seen_orig_list: List[set[int]] = []

        for sid in keys:
            prompt = np.asarray(prompts[sid], dtype=np.int64)
            mapped = [self.item_to_idx.get(int(item), 0) for item in prompt.tolist()]
            mapped = [m for m in mapped if m > 0]
            x_list.append(self._pad_left(mapped))
            seen_idx_list.append(set(mapped))
            seen_orig_list.append(set(int(i) for i in prompt.tolist()))

        x_batch = np.vstack(x_list).astype(np.int32)
        probs = self.model.predict(x_batch, batch_size=self.pred_batch_size, verbose=0)

        preds: Dict[int, np.ndarray] = {}
        for i, sid in enumerate(keys):
            row = probs[i].astype(np.float64)
            row[0] = -np.inf  # padding id

            if not self.pred_seen:
                for idx in seen_idx_list[i]:
                    row[idx] = -np.inf

            top_idx = np.argpartition(row, -min(top_k, len(row) - 1))[-min(top_k, len(row) - 1) :]
            top_idx = top_idx[np.argsort(row[top_idx])[::-1]]
            items = [self.idx_to_item[idx] for idx in top_idx if idx in self.idx_to_item]

            # Fill with popular items if needed.
            if len(items) < top_k:
                for item in self.popular_items:
                    if not self.pred_seen and item in seen_orig_list[i]:
                        continue
                    if item not in items:
                        items.append(item)
                    if len(items) >= top_k:
                        break

            preds[sid] = np.array(items[:top_k], dtype=np.int64)

        return preds

    def name(self) -> str:
        return self.model_name


class GRU4Rec(NeuralSequenceRecommender):
    def __init__(self, **kwargs) -> None:
        super().__init__(model_name="GRU4Rec", architecture="gru", **kwargs)


class BERT4Rec(NeuralSequenceRecommender):
    def __init__(self, mask_prob: float = 0.2, **kwargs) -> None:
        # mask_prob is accepted for API similarity, but not explicitly used in this compact notebook version.
        _ = mask_prob
        super().__init__(model_name="BERT4Rec", architecture="bert", **kwargs)


class SASRec(NeuralSequenceRecommender):
    def __init__(self, **kwargs) -> None:
        super().__init__(model_name="SASRec", architecture="sas", **kwargs)


@dataclass
class SimpleSessionDataset:
    input_data: pd.DataFrame
    train_data: pd.DataFrame
    test_data: pd.DataFrame
    test_prompts: Dict[int, np.ndarray]
    test_ground_truths: Dict[int, np.ndarray]
    n_withheld: int = 1

    @classmethod
    def from_sessions(
        cls,
        sessions_df: pd.DataFrame,
        test_frac: float = 0.2,
        n_withheld: int = 1,
        filter_non_trained_test_items: bool = True,
    ) -> "SimpleSessionDataset":
        df = sessions_df.copy()
        df["Time"] = pd.to_datetime(df["Time"])

        df = df.sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True)

        session_last = df.groupby("SessionId")["Time"].max().sort_values()
        n_sessions = len(session_last)
        n_test = max(1, math.ceil(test_frac * n_sessions))

        test_ids = set(session_last.index[-n_test:].astype(int).tolist())
        train_ids = set(session_last.index[:-n_test].astype(int).tolist())

        train_df = df[df["SessionId"].isin(train_ids)].copy()
        test_df = df[df["SessionId"].isin(test_ids)].copy()

        if filter_non_trained_test_items:
            train_items = set(train_df["ItemId"].astype(int).tolist())
            test_df = test_df[test_df["ItemId"].astype(int).isin(train_items)].copy()

        # Keep only test sessions that can produce at least one prediction case.
        lengths = test_df.groupby("SessionId").size()
        valid_test_sessions = lengths[lengths > n_withheld].index
        test_df = test_df[test_df["SessionId"].isin(valid_test_sessions)].copy()

        prompts: Dict[int, np.ndarray] = {}
        truths: Dict[int, np.ndarray] = {}
        for sid, grp in test_df.groupby("SessionId"):
            items = grp["ItemId"].to_numpy(dtype=np.int64)
            if len(items) <= n_withheld:
                continue
            prompts[int(sid)] = items[:-n_withheld]
            truths[int(sid)] = items[-n_withheld:]

        return cls(
            input_data=df,
            train_data=train_df.reset_index(drop=True),
            test_data=test_df.reset_index(drop=True),
            test_prompts=prompts,
            test_ground_truths=truths,
            n_withheld=n_withheld,
        )

    def get_train_data(self) -> pd.DataFrame:
        return self.train_data

    def get_test_prompts(self) -> Dict[int, np.ndarray]:
        return self.test_prompts

    def get_test_ground_truths(self) -> Dict[int, np.ndarray]:
        return self.test_ground_truths

    def get_unique_item_count(self) -> int:
        return int(self.input_data["ItemId"].nunique())

    def get_unique_sample_count(self) -> int:
        return int(self.input_data["SessionId"].nunique())

    def get_num_interactions(self) -> int:
        return int(len(self.input_data))

    def to_pickle(self, filepath: str) -> None:
        with open(filepath, "wb") as f:
            pickle.dump(self, f)


def evaluate_predictions(
    predictions: Dict[int, np.ndarray],
    ground_truths: Dict[int, np.ndarray],
    top_k: int,
    num_items: int,
    model_name: str,
    ) -> Dict[str, float]:
    common_ids = [sid for sid in ground_truths.keys() if sid in predictions]
    if len(common_ids) == 0:
        return {
            f"NDCG@{top_k}": 0.0,
            f"HitRate@{top_k}": 0.0,
            f"MRR@{top_k}": 0.0,
            f"Catalog coverage@{top_k}": 0.0,
        }

    ndcg_vals: List[float] = []
    hit_vals: List[float] = []
    mrr_vals: List[float] = []
    predicted_items: set[int] = set()

    for sid in common_ids:
        pred = np.asarray(predictions[sid], dtype=np.int64)[:top_k]
        gt = set(np.asarray(ground_truths[sid], dtype=np.int64).tolist())
        predicted_items.update(int(i) for i in pred.tolist())

        hit = 1.0 if any(item in gt for item in pred) else 0.0
        hit_vals.append(hit)

        rr = 0.0
        for rank, item in enumerate(pred, start=1):
            if int(item) in gt:
                rr = 1.0 / rank
                break
        mrr_vals.append(rr)

        dcg = 0.0
        for rank, item in enumerate(pred, start=1):
            if int(item) in gt:
                dcg += 1.0 / math.log2(rank + 1)

        ideal_hits = min(len(gt), top_k)
        idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
        ndcg_vals.append(dcg / idcg if idcg > 0 else 0.0)

    coverage = len(predicted_items) / max(1, num_items)

    return {
        f"NDCG@{top_k}": float(np.mean(ndcg_vals)),
        f"HitRate@{top_k}": float(np.mean(hit_vals)),
        f"MRR@{top_k}": float(np.mean(mrr_vals)),
        f"Catalog coverage@{top_k}": float(coverage),
    }


def download_movielens_100k(data_dir: Path, dataset_url: str) -> Path:
    data_dir.mkdir(parents=True, exist_ok=True)
    udata_path = data_dir / "ml-100k" / "u.data"
    if udata_path.exists():
        logging.info("Found existing MovieLens-100K at %s", udata_path)
        return udata_path

    archive_name = dataset_url.rsplit("/", 1)[-1]
    archive_path = data_dir / archive_name

    if not archive_path.exists():
        logging.info("Downloading %s", dataset_url)
        urllib.request.urlretrieve(dataset_url, archive_path)

    logging.info("Extracting %s", archive_path)
    if archive_path.suffix == ".zip":
        with zipfile.ZipFile(archive_path, "r") as zip_ref:
            zip_ref.extractall(data_dir)
    elif archive_name.endswith(".tar.gz") or archive_name.endswith(".tgz"):
        with tarfile.open(archive_path, "r:gz") as tar_ref:
            tar_ref.extractall(data_dir)
    else:
        raise ValueError(f"Unsupported archive format for {archive_path}")

    if not udata_path.exists():
        raise FileNotFoundError(f"Expected file not found at {udata_path}")
    return udata_path


def load_movielens_interactions(udata_path: Path) -> pd.DataFrame:
    return pd.read_csv(
        udata_path,
        sep="\t",
        names=["UserId", "ItemId", "Rating", "Timestamp"],
        engine="python",
    )


def sessionize(interactions: pd.DataFrame, session_gap_minutes: int, min_session_len: int) -> pd.DataFrame:
    if min_session_len < 2:
        raise ValueError("min_session_len must be at least 2")

    gap_seconds = session_gap_minutes * 60
    df = interactions.copy()
    df["Timestamp"] = df["Timestamp"].astype(int)
    df["ItemId"] = df["ItemId"].astype(int)
    df["Rating"] = df["Rating"].astype(float)

    df = df.sort_values(["UserId", "Timestamp", "ItemId"]).reset_index(drop=True)

    user_changed = df["UserId"].ne(df["UserId"].shift(1))
    ts_gap = df["Timestamp"].diff().fillna(0)
    new_session = user_changed | (ts_gap > gap_seconds)
    df["SessionId"] = new_session.cumsum().astype(int)

    session_sizes = df.groupby("SessionId").size()
    valid_sessions = session_sizes[session_sizes >= min_session_len].index
    df = df[df["SessionId"].isin(valid_sessions)].copy()

    id_map = {old_id: idx + 1 for idx, old_id in enumerate(sorted(df["SessionId"].unique()))}
    df["SessionId"] = df["SessionId"].map(id_map).astype(int)

    df["Time"] = pd.to_datetime(df["Timestamp"], unit="s", utc=True).dt.tz_localize(None)
    df["Time"] = df["Time"].dt.strftime("%Y-%m-%d %H:%M:%S.%f")
    df["Reward"] = df["Rating"].astype(float)

    out_df = df[["SessionId", "ItemId", "Time", "Reward"]]
    out_df = out_df.sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True)
    return out_df


def build_session_dataset(sessions_csv_path: Path, dataset_pickle_path: Path) -> SimpleSessionDataset:
    sessions_df = pd.read_csv(sessions_csv_path)
    dataset = SimpleSessionDataset.from_sessions(
        sessions_df,
        test_frac=TEST_FRAC,
        n_withheld=1,
        filter_non_trained_test_items=FILTER_NON_TRAINED_TEST_ITEMS,
    )
    dataset.to_pickle(str(dataset_pickle_path))
    return dataset


def instantiate_models() -> dict:
    models = {
        "Popular": SessionBasedPopular(),
        "SKNN": SessionBasedCF(
            k=SKNN_K,
            sample_size=SKNN_SAMPLE_SIZE,
            sampling="recent",
            similarity_measure="cosine",
            idf_weighting=True,
            filter_prompt_items=True,
        ),
        "S-SKNN": SessionBasedCF(
            k=SKNN_K,
            sample_size=SKNN_SAMPLE_SIZE,
            sampling="recent",
            similarity_measure="cosine",
            idf_weighting=True,
            sequential_weighting=True,
            filter_prompt_items=True,
        ),
        "SF-SKNN": SessionBasedCF(
            k=SKNN_K,
            sample_size=SKNN_SAMPLE_SIZE,
            sampling="recent",
            similarity_measure="cosine",
            idf_weighting=True,
            sequential_filter=True,
            filter_prompt_items=True,
        ),
        "V-SKNN": SessionBasedCF(
            k=SKNN_K,
            sample_size=SKNN_SAMPLE_SIZE,
            sampling="recent",
            similarity_measure="dot",
            idf_weighting=True,
            decay="harmonic",
            filter_prompt_items=True,
        ),
        "GRU4Rec": GRU4Rec(
            N=MAX_SEQ_LEN,
            emb_dim=NEURAL_EMB_DIM,
            hidden_dim=HIDDEN_DIM,
            drop_rate=DROP_RATE,
            num_epochs=NUM_EPOCHS,
            fit_batch_size=FIT_BATCH_SIZE,
            pred_batch_size=PRED_BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            pred_seen=False,
            verbose=VERBOSE,
        ),
        "BERT4Rec": BERT4Rec(
            N=MAX_SEQ_LEN,
            emb_dim=NEURAL_EMB_DIM,
            hidden_dim=HIDDEN_DIM,
            drop_rate=DROP_RATE,
            num_epochs=NUM_EPOCHS,
            fit_batch_size=FIT_BATCH_SIZE,
            pred_batch_size=PRED_BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            pred_seen=False,
            verbose=VERBOSE,
            mask_prob=MASK_PROB,
        ),
        "SASRec": SASRec(
            N=MAX_SEQ_LEN,
            emb_dim=NEURAL_EMB_DIM,
            hidden_dim=HIDDEN_DIM,
            drop_rate=DROP_RATE,
            num_epochs=NUM_EPOCHS,
            fit_batch_size=FIT_BATCH_SIZE,
            pred_batch_size=PRED_BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            pred_seen=False,
            verbose=VERBOSE,
        ),
    }
    return models


def run_single_model(model, dataset: SimpleSessionDataset, top_ks: list[int]) -> dict:
    model_name = model.name()
    max_top_k = max(top_ks)

    start_time = time.perf_counter()
    model.train(dataset.get_train_data())
    recs = model.predict(dataset.get_test_prompts(), top_k=max_top_k)
    elapsed = time.perf_counter() - start_time

    recs_path = RECS_DIR / f"recs_{model_name}.pickle"
    with open(recs_path, "wb") as f:
        pickle.dump(recs, f)

    row = {"Model": model_name, "TrainPredictSeconds": round(elapsed, 4)}
    for top_k in top_ks:
        metric_values = evaluate_predictions(
            predictions=recs,
            ground_truths=dataset.get_test_ground_truths(),
            top_k=top_k,
            num_items=dataset.get_unique_item_count(),
            model_name=model_name,
        )
        row.update(metric_values)

    return row

In [ ]:
udata_path = download_movielens_100k(DATA_DIR, DATASET_URL)
raw_interactions = load_movielens_interactions(udata_path)
sessions_df = sessionize(raw_interactions, SESSION_GAP_MINUTES, MIN_SESSION_LEN)

sessions_csv_path = DATA_DIR / "ml100k_sessions.csv"
sessions_df.to_csv(sessions_csv_path, index=False)

dataset_pickle_path = DATA_DIR / "ml100k_session_dataset.pickle"
dataset = build_session_dataset(sessions_csv_path, dataset_pickle_path)

print("Interactions:", dataset.get_num_interactions())
print("Sessions:", dataset.get_unique_sample_count())
print("Unique items:", dataset.get_unique_item_count())
print("Dataset pickle:", dataset_pickle_path)

In [ ]:
all_models = instantiate_models()
rows = []
failures = []

for model_name in INCLUDE_MODELS:
    if model_name not in all_models:
        failures.append({"Model": model_name, "Error": "Unknown model name"})
        continue

    model = all_models[model_name]
    logging.info("Starting model %s", model_name)
    try:
        row = run_single_model(model, dataset, TOP_KS)
        rows.append(row)
        logging.info("Finished model %s", model_name)
    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"
        failures.append({"Model": model_name, "Error": error_message})
        logging.exception("Model %s failed", model_name)

results_df = pd.DataFrame(rows)
if not results_df.empty:
    primary_col = f"NDCG@{max(TOP_KS)}"
    if primary_col in results_df.columns:
        results_df = results_df.sort_values(primary_col, ascending=False)

failures_df = pd.DataFrame(failures)

results_csv = RESULTS_DIR / "baseline_results_ml100k.csv"
failures_csv = RESULTS_DIR / "baseline_failures_ml100k.csv"
results_df.to_csv(results_csv, index=False)
failures_df.to_csv(failures_csv, index=False)

print("Saved results:", results_csv)
print("Saved failures:", failures_csv)
print("Saved recommendations dir:", RECS_DIR)

In [ ]:
display(results_df)
if len(failures_df) > 0:
    display(failures_df)

Tip: for quick neural smoke-tests set `INCLUDE_MODELS = ["GRU4Rec", "BERT4Rec", "SASRec"]` and `NUM_EPOCHS = 2`, then run all cells.